In [1]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

In [2]:
# ---------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------
df = pd.read_csv("Task1_wildfire_weather_preprocessed_Znormalized_v0.1.csv")

# Extract year for train/test split
df['year'] = pd.to_datetime(df['year_month']).dt.year
panel_df = df[(df["year"] >= 2018) & (df["year"] <= 2021)].copy()

# Features (exclude id columns)
feature_cols = [
    "lat", "lon",
    "month_sin", "month_cos",
    "avg_tmax_c", "avg_tmin_c", "tot_prcp_mm"
]


train_df = panel_df[panel_df["year"] <= 2020].copy()
val_df   = panel_df[panel_df["year"] == 2021].copy()



X_train = train_df[feature_cols].values
y_train = train_df["has_fire"].values

X_val = val_df[feature_cols].values
y_val = val_df["has_fire"].values

In [3]:
from sklearn.linear_model import LogisticRegression
# model = LogisticRegression(max_iter=500, class_weight="balanced")

from sklearn.neural_network import MLPClassifier
# model = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300)

from sklearn.svm import SVC
# model = SVC(
#     kernel="rbf",
#     probability=True,
#     class_weight="balanced",
#     C=1.0,
#     gamma="scale"
# )

from sklearn.ensemble import RandomForestClassifier
# model = RandomForestClassifier(
#     n_estimators=300,
#     max_depth=None,
#     min_samples_split=2,
#     min_samples_leaf=1,
#     class_weight="balanced",
#     n_jobs=-1,
#     random_state=42)

from sklearn.ensemble import ExtraTreesClassifier
# model = ExtraTreesClassifier(
#     n_estimators=300,
#     max_depth=None,
#     min_samples_split=2,
#     min_samples_leaf=1,
#     class_weight="balanced",
#     n_jobs=-1,
#     random_state=42)

from sklearn.naive_bayes import GaussianNB
# model = GaussianNB()

import xgboost as xgb
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
    n_jobs=-1,
    random_state=42)


def risk_at_k_percent(risk, y_true, k):
    """
    risk: 模型输出的 risk score (numpy array)
    y_true: 真实标签 (numpy array)
    k: 百分比，例如 0.01, 0.03, 0.05
    """
    df = pd.DataFrame({"risk": risk, "y": y_true})
    df = df.sort_values("risk", ascending=False)

    top_k = int(len(df) * k)
    top_k = max(top_k, 1)  # 防止太小

    captured = df.head(top_k)["y"].sum()
    total = df["y"].sum()

    return captured / total if total > 0 else 0


In [4]:
model.fit(X_train, y_train)
risk = model.predict_proba(X_val)[:, 1]
print("AUC =", roc_auc_score(y_val, risk))

AUC = 0.815765486501758


In [5]:
print("risk@1% =", risk_at_k_percent(risk, y_val, 0.01))
print("risk@3% =", risk_at_k_percent(risk, y_val, 0.03))
print("risk@5% =", risk_at_k_percent(risk, y_val, 0.05))

risk@1% = 0.11262798634812286
risk@3% = 0.22184300341296928
risk@5% = 0.3310580204778157


In [6]:
df_2023 = df[df["year"] == 2023].copy()
X_2023 = df_2023[feature_cols].values
risk_2023 = model.predict_proba(X_2023)[:, 1]
risk_2023
# df_2023["risk"] = risk_2023
# df_2023.head()

array([0.09256133, 0.02054658, 0.01900867, 0.06109399, 0.1026503 ,
       0.12246063, 0.19060367, 0.5058518 , 0.68910134, 0.42187008,
       0.5495238 , 0.8474552 , 0.61319035, 0.30766004, 0.7436187 ,
       0.17001721, 0.5526311 , 0.6281844 , 0.5144368 , 0.2158621 ,
       0.7937571 , 0.61305887, 0.9702256 , 0.965175  , 0.94779825,
       0.8393594 , 0.88839036, 0.85276204, 0.7122919 , 0.84915787,
       0.40147606, 0.7415281 , 0.8429629 , 0.88493717, 0.89773405,
       0.72950214, 0.42811418, 0.89367497, 0.83531123, 0.74316317,
       0.8593811 , 0.8911833 , 0.3698901 , 0.7388951 , 0.08970236,
       0.12066542, 0.8354164 , 0.27522665, 0.48214304, 0.8103195 ,
       0.8702529 , 0.6986258 , 0.846339  , 0.8823731 , 0.49302596,
       0.68888664, 0.78458124, 0.925688  , 0.80748904, 0.92461854,
       0.4923101 , 0.8622524 , 0.73703456, 0.8485154 , 0.5142072 ,
       0.72435707, 0.6967442 , 0.66533315, 0.7261656 , 0.84181935,
       0.7039394 , 0.6816705 , 0.62772864, 0.765353  , 0.88220

## Clustering

In [7]:
panel = df[(df["year"] >= 2018) & (df["year"] <= 2021)].copy()
pos = panel[panel["has_fire"] == 1].copy()

feature_cols = [
    "lat", "lon",
    "month_sin", "month_cos",
    "avg_tmax_c", "avg_tmin_c", "tot_prcp_mm"
]

X_pos = pos[feature_cols].values


In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

scaler = StandardScaler()
X_pos_scaled = scaler.fit_transform(X_pos)

K=5
kmeans = KMeans(n_clusters=K, random_state=42)
kmeans.fit(X_pos_scaled)

centers = kmeans.cluster_centers_


In [9]:
def cluster_risk(X_raw):
    X_scaled = scaler.transform(X_raw)
    # 距离所有中心
    dists = np.linalg.norm(
        X_scaled[:, None, :] - centers[None, :, :],
        axis=2
    )  # shape: [n_samples, K]
    d_min = dists.min(axis=1)
    risk = 1.0 / (1.0 + d_min)  # 或 np.exp(-alpha * d_min)
    return risk


In [10]:
X_panel = panel[feature_cols].values
risk_panel = cluster_risk(X_panel)
y_panel = panel["has_fire"].values

print("AUC (cluster risk) =", roc_auc_score(y_panel, risk_panel))

AUC (cluster risk) = 0.6760222534205174


In [11]:
print("risk@1% =", risk_at_k_percent(risk_panel, y_panel, 0.01))
print("risk@3% =", risk_at_k_percent(risk_panel, y_panel, 0.03))
print("risk@5% =", risk_at_k_percent(risk_panel, y_panel, 0.05))

risk@1% = 0.014925373134328358
risk@3% = 0.06301824212271974
risk@5% = 0.1077943615257048


In [12]:
df_2023 = df[df["year"] == 2023].copy()
X_2023 = df_2023[feature_cols].values
risk_2023 = model.predict_proba(X_2023)[:, 1]
risk_2023

array([0.09256133, 0.02054658, 0.01900867, 0.06109399, 0.1026503 ,
       0.12246063, 0.19060367, 0.5058518 , 0.68910134, 0.42187008,
       0.5495238 , 0.8474552 , 0.61319035, 0.30766004, 0.7436187 ,
       0.17001721, 0.5526311 , 0.6281844 , 0.5144368 , 0.2158621 ,
       0.7937571 , 0.61305887, 0.9702256 , 0.965175  , 0.94779825,
       0.8393594 , 0.88839036, 0.85276204, 0.7122919 , 0.84915787,
       0.40147606, 0.7415281 , 0.8429629 , 0.88493717, 0.89773405,
       0.72950214, 0.42811418, 0.89367497, 0.83531123, 0.74316317,
       0.8593811 , 0.8911833 , 0.3698901 , 0.7388951 , 0.08970236,
       0.12066542, 0.8354164 , 0.27522665, 0.48214304, 0.8103195 ,
       0.8702529 , 0.6986258 , 0.846339  , 0.8823731 , 0.49302596,
       0.68888664, 0.78458124, 0.925688  , 0.80748904, 0.92461854,
       0.4923101 , 0.8622524 , 0.73703456, 0.8485154 , 0.5142072 ,
       0.72435707, 0.6967442 , 0.66533315, 0.7261656 , 0.84181935,
       0.7039394 , 0.6816705 , 0.62772864, 0.765353  , 0.88220